In [ ]:
# !pip install flask requests PyPDF2 spacy pdfplumber
# !python -m spacy download en_core_web_sm

# 📄 Resume Parser API — Integration Guide

### 🧩 Overview

This API extracts text from a given PDF resume URL and returns structured data in JSON format.
It’s a Flask backend endpoint running locally (`localhost:5000`) for development and testing.

---

## 🚀 Endpoint Details

**Base URL:**

```
http://127.0.0.1:5000/parse_resume
```

**Method:**
`POST`

**Headers:**

```
Content-Type: application/json
```

**Request Body Example:**

```json
{
  "url": "https://careercenter.ucdavis.edu/sites/g/files/dgvnsk15461/files/media/documents/GEN-Beginning-Resume.pdf"
}
```

---

## ⚛️ Frontend Integration (React + Axios)

```javascript
import axios from "axios";

export const parseResume = async (pdfUrl) => {
  try {
    const response = await axios.post("http://127.0.0.1:5000/parse_resume", {
      url: pdfUrl,
    }, {
      headers: { "Content-Type": "application/json" },
    });

    console.log("Resume Parsed Successfully:", response.data);
    return response.data;
  } catch (error) {
    console.error("Error Parsing Resume:", error.response?.data || error.message);
  }
};

// Example Usage:
parseResume("https://careercenter.ucdavis.edu/sites/g/files/dgvnsk15461/files/media/documents/GEN-Beginning-Resume.pdf");
```

---

## 🧪 Example (Postman Test)

Below is the Postman test demonstration for the same request.

![Postman Example](https://res.cloudinary.com/dah7l8utl/image/upload/v1762014929/Screenshot_2025-11-01_220411_lnrsdx.png)

---

## 📦 Sample Response

```json
{
  "status": "success",
  "url": "https://careercenter.ucdavis.edu/sites/g/files/dgvnsk15461/files/media/documents/GEN-Beginning-Resume.pdf",
  "resume_json": {
    "name": "Detected Name (Sample)",
    "email": "detected.email@example.com",
    "skills": ["Python", "Flask", "Machine Learning"],
    "summary": "Analyn Ocampo Davis, CA 95616 | (559) 555-5683 Email: aocampo@ucdavis.edu OBJECTIVE Work study eligible undergraduate student seeking an on-campus job opportunity. EDUCATION Intended Major: Communication, Bachelor of Arts Degree University of California, Davis Expected Graduation: June 2027 ..."
  }
}
```


In [ ]:
from flask import Flask, request, jsonify
import spacy
import pdfplumber
import re
import io
from PyPDF2 import PdfReader

app = Flask(__name__)
nlp = spacy.load("en_core_web_sm")

def extract_text_from_pdf(file_stream):
    """Extract all text from PDF file."""
    text = ""
    with pdfplumber.open(file_stream) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text.strip()

def extract_email(text):
    """Extract email using regex."""
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else None

def extract_phone(text):
    """Extract phone numbers."""
    match = re.search(r'(\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4})', text)
    return match.group(0) if match else None

def extract_name(doc):
    """Use spaCy NER to get likely candidate for name."""
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    # fallback: first two words of resume
    lines = [l.strip() for l in doc.text.split("\n") if l.strip()]
    if len(lines) > 0:
        first_line = lines[0]
        return " ".join(first_line.split()[:2])
    return None

def extract_skills(text):
    """Extract skills using keyword matching."""
    skill_keywords = [
        "Python", "Flask", "Machine Learning", "Communication", "Word", "Outlook",
        "Mac", "PC", "Internet", "Tagalog", "Email"
    ]
    found = [s for s in skill_keywords if re.search(rf'\b{s}\b', text, re.IGNORECASE)]
    return list(set(found))

def extract_education(text):
    """Basic education detection."""
    edu = re.findall(r'(University of [A-Z][a-z]+(?: [A-Z][a-z]+)*)|([A-Z][a-z]+ University)', text)
    edu_list = [e[0] or e[1] for e in edu if e[0] or e[1]]
    return list(set(edu_list))

def extract_experience(text):
    """Find company or experience section lines."""
    experience_lines = []
    for line in text.split("\n"):
        if re.search(r'(Receptionist|Provider|Intern|Engineer|Developer|Manager)', line, re.IGNORECASE):
            experience_lines.append(line.strip())
    return experience_lines

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        "message": "Welcome to Resume Parser API",
        "usage": {
            "POST /parse": "Upload a PDF resume in form-data with key='file'"
        }
    })

@app.route('/parse', methods=['POST'])
def parse_resume():
    if 'file' not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    file = request.files['file']
    text = extract_text_from_pdf(file)

    doc = nlp(text)
    resume_data = {
        "name": extract_name(doc),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "skills": extract_skills(text),
        "education": extract_education(text),
        "experience": extract_experience(text),
        "summary": text[:400] + "..."
    }

    return jsonify({"status": "success", "resume_json": resume_data})


if __name__ == "__main__":
    app.run(port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
